In [1]:
# # 必要なパッケージの読み込み
# library(caret)
# library(Metrics)
# library(ggplot2)
# library(lattice)
# library(pls)

# # 対象ファイル名一覧
# file_names <- c(
#   "n_base.csv", "n_base_OH.csv", "n_base_FP.csv",
#   "n_all.csv", "n_all_OH.csv", "n_all_FP.csv",
#   "m_base.csv", "m_base_OH.csv", "m_base_FP.csv",
#   "m_all.csv", "m_all_OH.csv", "m_all_FP.csv"
# )

# # データパスの共通部分
# base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/"

# # モデルの出力用日付
# today <- format(Sys.Date(), "%Y%m%d")

# # 評価結果の格納用データフレームの初期化
# target_vars <- c("Jsc", "Voc", "FF", "PCEmax")
# metric_names <- c("Best_ncomp", "R2", "RMSE", "MAE", "RPD", "n_samples", "n_features")
# result_matrix <- matrix(nrow = length(metric_names) * length(target_vars), ncol = length(file_names))
# rownames(result_matrix) <- as.vector(t(outer(metric_names, target_vars, paste, sep = "_")))
# colnames(result_matrix) <- file_names

# #--- メインループ ---#
# for (file in file_names) {
#   filepath <- paste0(base_path, file)
#   cat("\n=== Processing:", file, "===\n")
#   df_all <- read.csv(filepath)
#   cat("Final dataset size:", dim(df_all)[1], dim(df_all)[2], "\n")

#   feature_vars <- setdiff(colnames(df_all), target_vars)

#   for (target_var in target_vars) {
#     cat("\n---\nTraining model for:", target_var, "on", file, "\n")
#     df <- df_all[, c(feature_vars, target_var)]
#     df <- df[complete.cases(df), ]

#     # モデルチューニング
#     tune_grid <- expand.grid(ncomp = 1:15)
#     ctrl <- trainControl(method = "cv", number = 10)

#     model <- train(
#       formula(paste(target_var, "~ .")),
#       data = df,
#       method = "pls",
#       metric = "RMSE",
#       trControl = ctrl,
#       tuneGrid = tune_grid
#     )

#     pred <- predict(model, df)
#     obs <- df[[target_var]]

#     R2 <- round(cor(pred, obs)^2, 3)
#     RMSE_val <- round(rmse(obs, pred), 3)
#     MAE_val <- round(mae(obs, pred), 3)
#     RPD_val <- round(sd(obs) / RMSE_val, 3)

#     cat("Best ncomp value:", model$bestTune$ncomp, "\n")
#     cat(file, target_var, ": R2 =", R2, ", RMSE =", RMSE_val, ", MAE =", MAE_val, ", RPD =", RPD_val, "\n")

#     # 結果保存
#     result_matrix[paste0("Best_ncomp_", target_var), file] <- model$bestTune$ncomp
#     result_matrix[paste0("R2_", target_var), file] <- R2
#     result_matrix[paste0("RMSE_", target_var), file] <- RMSE_val
#     result_matrix[paste0("MAE_", target_var), file] <- MAE_val
#     result_matrix[paste0("RPD_", target_var), file] <- RPD_val
#     result_matrix[paste0("n_samples_", target_var), file] <- nrow(df)
#     result_matrix[paste0("n_features_", target_var), file] <- ncol(df) - 1

#     # 寄与度（VIP）の保存
#     vip_vals <- varImp(model, scale = TRUE)$importance
#     vip_file <- paste0("new", today, "_VIP_PLS_", tools::file_path_sans_ext(file), "_", target_var, ".csv")
#     write.csv(vip_vals, vip_file)

#     # 軸スケール決定関数
#     get_axis_limit <- function(values) {
#       max_val <- max(values, na.rm = TRUE)
#       if (max_val <= 1.5) {
#         return(ceiling(max_val / 0.1) * 0.1)
#       } else if (max_val <= 5) {
#         return(ceiling(max_val / 0.5) * 0.5)
#       } else if (max_val <= 30) {
#         return(ceiling(max_val / 2) * 2)
#       } else {
#         return(ceiling(max_val / 5) * 5)
#       }
#     }

#     range_max <- get_axis_limit(c(obs, pred))

#     # プロット
#     p <- ggplot(data.frame(Predicted = pred, Observed = obs), aes(x = Observed, y = Predicted)) +
#       geom_point(color = "blue", alpha = 0.7) +
#       geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "red") +
#       scale_x_continuous(limits = c(0, range_max)) +
#       scale_y_continuous(limits = c(0, range_max)) +
#       coord_fixed(ratio = 1) +
#       labs(
#         title = paste0(target_var, " Prediction (", file, ")"),
#         x = "Observed",
#         y = "Predicted"
#       ) +
#       theme_minimal() +
#       theme(
#         plot.title = element_text(hjust = 0.5, size = 14),
#         plot.subtitle = element_text(hjust = 0.5, size = 10)
#       ) +
#       annotate("text", x = range_max * 0.05, y = range_max * 0.95, hjust = 0, vjust = 1, size = 4,
#                label = paste0("RMSE = ", RMSE_val, "\nMAE = ", MAE_val, "\nRPD = ", RPD_val)) +
#       annotate("text", x = range_max * 0.95, y = range_max * 0.05, hjust = 1, vjust = 0, size = 5,
#                label = paste0("R² = ", R2))

#     # モデル＋データ保存
#     model_data_bundle <- list(
#       model = model,
#       training_data = df
#     )
#     rds_file <- paste0("new", today, "_model_data_PLS_", tools::file_path_sans_ext(file), "_", target_var, ".rds")
#     saveRDS(model_data_bundle, file = rds_file)

#     # PDF出力
#     plot_file <- paste0("new", today, "_plot_PLS_", tools::file_path_sans_ext(file), "_", target_var, ".pdf")
#     pdf(plot_file, width = 6, height = 6)
#     print(p)
#     dev.off()
#   }
# }

# #--- 結果の保存 ---#
# output_file <- paste0("new", today, "_PLS_summary_", today, ".csv")
# write.csv(result_matrix, output_file, row.names = TRUE)
# cat("\nSummary saved as", output_file, "\n")


Loading required package: ggplot2

Loading required package: lattice


Attaching package: 'Metrics'


The following objects are masked from 'package:caret':

    precision, recall



Attaching package: 'pls'


The following object is masked from 'package:caret':

    R2


The following object is masked from 'package:stats':

    loadings





=== Processing: n_base.csv ===
Final dataset size: 358 12 

---
Training model for: Jsc on n_base.csv 


Warning message:
"predictions failed for Fold01: ncomp=15 Error in object$coefficients[, , ncomp, drop = FALSE] : 
  subscript out of bounds
"
Warning message:
"predictions failed for Fold02: ncomp=15 Error in object$coefficients[, , ncomp, drop = FALSE] : 
  subscript out of bounds
"
Warning message:
"predictions failed for Fold03: ncomp=15 Error in object$coefficients[, , ncomp, drop = FALSE] : 
  subscript out of bounds
"
Warning message:
"predictions failed for Fold04: ncomp=15 Error in object$coefficients[, , ncomp, drop = FALSE] : 
  subscript out of bounds
"
Warning message:
"predictions failed for Fold05: ncomp=15 Error in object$coefficients[, , ncomp, drop = FALSE] : 
  subscript out of bounds
"
Warning message:
"predictions failed for Fold06: ncomp=15 Error in object$coefficients[, , ncomp, drop = FALSE] : 
  subscript out of bounds
"
Warning message:
"predictions failed for Fold07: ncomp=15 Error in object$coefficients[, , ncomp, drop = FALSE] : 
  subscript out of bounds
"

Something is wrong; all the RMSE metric values are missing:
      RMSE        Rsquared        MAE     
 Min.   : NA   Min.   : NA   Min.   : NA  
 1st Qu.: NA   1st Qu.: NA   1st Qu.: NA  
 Median : NA   Median : NA   Median : NA  
 Mean   :NaN   Mean   :NaN   Mean   :NaN  
 3rd Qu.: NA   3rd Qu.: NA   3rd Qu.: NA  
 Max.   : NA   Max.   : NA   Max.   : NA  
 NA's   :15    NA's   :15    NA's   :15   


ERROR: Error: Stopping


In [3]:
# # --- 必要なパッケージ ---
# library(caret)
# library(pls)
# library(Metrics)
# library(ggplot2)

# # --- 対象ファイルなどの設定 ---
# file_names <- c(
#   "n_base.csv", "n_base_OH.csv", "n_base_FP.csv",
#   "n_all.csv", "n_all_OH.csv", "n_all_FP.csv",
#   "m_base.csv", "m_base_OH.csv", "m_base_FP.csv",
#   "m_all.csv", "m_all_OH.csv", "m_all_FP.csv"
# )

# base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/"
# target_vars <- c("Jsc", "Voc", "FF", "PCEmax")
# metric_names <- c("Best_ncomp", "R2", "RMSE", "MAE", "RPD", "Q2", "n_samples", "n_features")
# result_matrix <- matrix(nrow = length(metric_names) * length(target_vars), ncol = length(file_names))
# rownames(result_matrix) <- as.vector(t(outer(metric_names, target_vars, paste, sep = "_")))
# colnames(result_matrix) <- file_names

# today <- format(Sys.Date(), "%Y%m%d")

# # --- VIP計算関数 ---
# calc_vip <- function(pls_model, ncomp) {
#   max_comp <- min(ncomp, nrow(pls_model$Yloadings))
#   SS <- pls_model$Yloadings[1:max_comp, , drop = FALSE]^2
#   W <- pls_model$loading.weights[, 1:max_comp, drop = FALSE]
#   p <- nrow(W)
#   Wnorm2 <- colSums(W^2)
#   SSY <- sum(SS)

#   vip_scores <- numeric(p)
#   for (j in 1:p) {
#     weight <- 0
#     for (a in 1:max_comp) {
#       if (Wnorm2[a] == 0) next
#       weight <- weight + SS[a] * (W[j, a]^2 / Wnorm2[a])
#     }
#     vip_scores[j] <- sqrt(p * weight / SSY)
#   }
#   names(vip_scores) <- rownames(W)
#   return(vip_scores)
# }

# # --- メインループ ---
# for (file in file_names) {
#   filepath <- paste0(base_path, file)
#   cat("\n=== Processing:", file, "===\n")
#   df_all <- read.csv(filepath)
#   feature_vars <- setdiff(colnames(df_all), target_vars)

#   for (target_var in target_vars) {
#     cat("\n---\nTraining model for:", target_var, "on", file, "\n")
#     df <- df_all[, c(feature_vars, target_var)]
#     df <- df[complete.cases(df), ]

#     max_ncomp <- min(10, ncol(df) - 1, nrow(df) - 1)
#     if (max_ncomp < 1) {
#       cat("  Skipping due to insufficient data.\n")
#       next
#     }

#     tune_grid <- expand.grid(ncomp = 1:max_ncomp)
#     ctrl <- trainControl(method = "cv", number = 10, returnResamp = "final")

#     model <- train(
#       formula(paste(target_var, "~ .")),
#       data = df,
#       method = "pls",
#       metric = "RMSE",
#       trControl = ctrl,
#       tuneGrid = tune_grid,
#       preProcess = c("center", "scale")
#     )

#     pred <- predict(model, df)
#     obs <- df[[target_var]]
#     R2 <- round(cor(pred, obs)^2, 3)
#     RMSE_val <- round(rmse(obs, pred), 3)
#     MAE_val <- round(mae(obs, pred), 3)
#     RPD_val <- round(sd(obs) / RMSE_val, 3)

#     # PRESSおよびQ2計算
#     pls_model <- model$finalModel
#     press <- sum((obs - pred)^2)
#     tss <- sum((obs - mean(obs))^2)
#     Q2_val <- round(1 - press / tss, 3)

#     best_ncomp <- model$bestTune$ncomp

#     # 結果格納
#     result_matrix[paste0("Best_ncomp_", target_var), file] <- best_ncomp
#     result_matrix[paste0("R2_", target_var), file] <- R2
#     result_matrix[paste0("RMSE_", target_var), file] <- RMSE_val
#     result_matrix[paste0("MAE_", target_var), file] <- MAE_val
#     result_matrix[paste0("RPD_", target_var), file] <- RPD_val
#     result_matrix[paste0("Q2_", target_var), file] <- Q2_val
#     result_matrix[paste0("n_samples_", target_var), file] <- nrow(df)
#     result_matrix[paste0("n_features_", target_var), file] <- ncol(df) - 1

#     # モデル＋データ保存
#     model_data_bundle <- list(
#       model = model,
#       training_data = df
#     )
#     rds_file <- paste0("new20250624_model_data_", tools::file_path_sans_ext(file), "_", target_var, "_PLS_", today, ".rds")
#     saveRDS(model_data_bundle, file = rds_file)

#     # VIP保存
#     vip <- calc_vip(pls_model, best_ncomp)
#     vip_df <- data.frame(Variable = names(vip), VIP = vip)
#     vip_file <- paste0("new20250624_vip_", tools::file_path_sans_ext(file), "_", target_var, "_PLS_", today, ".csv")
#     write.csv(vip_df, vip_file, row.names = FALSE)

#     # プロット
#     range_max <- ceiling(max(c(obs, pred)) * 1.1)
#     p <- ggplot(data.frame(Predicted = pred, Observed = obs), aes(x = Observed, y = Predicted)) +
#       geom_point(color = "blue", alpha = 0.7) +
#       geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "red") +
#       scale_x_continuous(limits = c(0, range_max)) +
#       scale_y_continuous(limits = c(0, range_max)) +
#       coord_fixed(ratio = 1) +
#       labs(
#         title = paste0(target_var, " Prediction (", file, ")"),
#         x = "Observed",
#         y = "Predicted"
#       ) +
#       theme_minimal() +
#       annotate("text", x = range_max * 0.05, y = range_max * 0.95, hjust = 0, vjust = 1, size = 4,
#                label = paste0("RMSE = ", RMSE_val, "\nMAE = ", MAE_val, "\nRPD = ", RPD_val)) +
#       annotate("text", x = range_max * 0.95, y = range_max * 0.05, hjust = 1, vjust = 0, size = 5,
#                label = paste0("R² = ", R2, "\nQ² = ", Q2_val))

#     plot_file <- paste0("new20250624_plot_", tools::file_path_sans_ext(file), "_", target_var, "_PLS_", today, ".pdf")
#     pdf(plot_file, width = 6, height = 6)
#     print(p)
#     dev.off()
#   }
# }

# # --- 結果出力 ---
# output_file <- paste0("new20250624_pls_summary_", today, ".csv")
# write.csv(result_matrix, output_file, row.names = TRUE)
# cat("\nSummary saved as", output_file, "\n")



=== Processing: n_base.csv ===

---
Training model for: Jsc on n_base.csv 

---
Training model for: Voc on n_base.csv 

---
Training model for: FF on n_base.csv 

---
Training model for: PCEmax on n_base.csv 

=== Processing: n_base_OH.csv ===

---
Training model for: Jsc on n_base_OH.csv 


Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: SMILESsnamenM_ICMM, SMILESsnamep1M_BDTH8, SMILESsnamep1M_DTSCEH, SMILESsnamep1M_DTTPTP4, SMILESsnamep1M_PBDTTS3, SMILESsnamep1M_PTzBTHD"
Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: SMILESsnamep1M_DTSTCh, SMILESsnamep1M_DTTPTP3, SMILESsnamep1M_DTTPTP5, SMILESsnamep1M_PBDTTS2, SMILESsnamep1M_PCDTDPP, SMILESsnamep1M_PTB2, SMILESsnamep1M_PTzQT14"
Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: SMILESsnamenM_ICEM, SMILESsnamenM_TC61BM, SMILESsnamep1M_BDTH6, SMILESsnamep1M_BDTH7, SMILESsnamep1M_BDTT6, SMILESsnamep1M_DTSCh, SMILESsnamep1M_DTSChy, SMILESsnamep1M_DTSTChy, SMILESsnamep1M_PDPP2FTC12, SMILESsnamep1M_PTBDTffDTBT"
Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uni


---
Training model for: Voc on n_base_OH.csv 


Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: SMILESsnamenM_TC61PM, SMILESsnamep1M_BDTH3, SMILESsnamep1M_DTSCEH, SMILESsnamep1M_DTSTChy, SMILESsnamep1M_DTTPTP1, SMILESsnamep1M_PBDTTEHDTEHBTffP2, SMILESsnamep1M_PBDTTS3, SMILESsnamep1M_PCDTPX, SMILESsnamep1M_PDPP2FTC14, SMILESsnamep1M_Poly.N9.fheptadecanyl2.7carbazolealt5.5.4.f.7.fdi2thienyl.pyrazino.2.3d.pyridazine.P3"
Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: SMILESsnamep1M_BDTH1, SMILESsnamep1M_BDTH8, SMILESsnamep1M_BDTH9, SMILESsnamep1M_BDTT6, SMILESsnamep1M_PBDT.DTBT, SMILESsnamep1M_Poly.N9.fheptadecanyl2.7carbazolealt5.5.4.f.7.fdi2thienyl.2.3dioctylpyrazino.2.3d.pyridazine.P2"
Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: SMILESsnamep1M_DTSC6, SMILESsnamep1M_DTSTCh, SMILESsnamep1M


---
Training model for: FF on n_base_OH.csv 


...


---
Training model for: FF on m_all_FP.csv 


Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: X17.1"
Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: X111.3"
Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: X46.4"



---
Training model for: PCEmax on m_all_FP.csv 


Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: X46.4"
Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: X17.1"
Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: X111.3"
Warning message:
"Removed 10 rows containing missing values or values outside the scale range
(`geom_point()`)."



Summary saved as new20250624_pls_summary_20250624.csv 


In [4]:
# --- 必要なパッケージ ---
library(caret)
library(pls)
library(Metrics)
library(ggplot2)

# --- 対象ファイルなどの設定 ---
file_names <- c(
  "n_base.csv", "n_base_OH.csv", "n_base_FP.csv",
  "n_all.csv", "n_all_OH.csv", "n_all_FP.csv",
  "m_base.csv", "m_base_OH.csv", "m_base_FP.csv",
  "m_all.csv", "m_all_OH.csv", "m_all_FP.csv"
)

base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/"
target_vars <- c("Jsc", "Voc", "FF", "PCEmax")
metric_names <- c("Best_ncomp", "R2", "RMSE", "MAE", "RPD", "Q2", "n_samples", "n_features")
result_matrix <- matrix(nrow = length(metric_names) * length(target_vars), ncol = length(file_names))
rownames(result_matrix) <- as.vector(t(outer(metric_names, target_vars, paste, sep = "_")))
colnames(result_matrix) <- file_names

today <- format(Sys.Date(), "%Y%m%d")

# --- VIP計算関数 ---
calc_vip <- function(pls_model, ncomp) {
  max_comp <- min(ncomp, nrow(pls_model$Yloadings))
  SS <- pls_model$Yloadings[1:max_comp, , drop = FALSE]^2
  W <- pls_model$loading.weights[, 1:max_comp, drop = FALSE]
  p <- nrow(W)
  Wnorm2 <- colSums(W^2)
  SSY <- sum(SS)

  vip_scores <- numeric(p)
  for (j in 1:p) {
    weight <- 0
    for (a in 1:max_comp) {
      if (Wnorm2[a] == 0) next
      weight <- weight + SS[a] * (W[j, a]^2 / Wnorm2[a])
    }
    vip_scores[j] <- sqrt(p * weight / SSY)
  }
  names(vip_scores) <- rownames(W)
  return(vip_scores)
}

# --- メインループ ---
for (file in file_names) {
  filepath <- paste0(base_path, file)
  cat("\n=== Processing:", file, "===\n")
  df_all <- read.csv(filepath)
  feature_vars <- setdiff(colnames(df_all), target_vars)

  for (target_var in target_vars) {
    cat("\n---\nTraining model for:", target_var, "on", file, "\n")
    df <- df_all[, c(feature_vars, target_var)]
    df <- df[complete.cases(df), ]

    max_ncomp <- min(10, ncol(df) - 1, nrow(df) - 1)
    if (max_ncomp < 1) {
      cat("  Skipping due to insufficient data.\n")
      next
    }

    tune_grid <- expand.grid(ncomp = 1:max_ncomp)
    ctrl <- trainControl(method = "cv", number = 10, savePredictions = "final")

    model <- train(
      formula(paste(target_var, "~ .")),
      data = df,
      method = "pls",
      metric = "RMSE",
      trControl = ctrl,
      tuneGrid = tune_grid,
      preProcess = c("center", "scale")
    )

    # 交差検証予測値の取得とフィルタリング（安全に）
    cv_preds <- model$pred
    best_params <- model$bestTune

    # bestTuneのパラメータだけ残っている列でフィルタ
    for (param in names(best_params)) {
      if (param %in% colnames(cv_preds)) {
        cv_preds <- cv_preds[cv_preds[[param]] == best_params[[param]], ]
      }
    }

    # 値が空でないか確認
    if (nrow(cv_preds) > 0) {
      pred <- cv_preds$pred
      obs <- cv_preds$obs

      R2 <- round(cor(obs, pred)^2, 3)
      RMSE_val <- round(rmse(obs, pred), 3)
      MAE_val <- round(mae(obs, pred), 3)
      RPD_val <- round(sd(obs) / RMSE_val, 3)
    } else {
      warning("No predictions matched bestTune. Skipping this model evaluation.")
      R2 <- RMSE_val <- MAE_val <- RPD_val <- NA
    }


    # PRESSおよびQ2計算
    pls_model <- model$finalModel
    press <- sum((obs - pred)^2)
    tss <- sum((obs - mean(obs))^2)
    Q2_val <- round(1 - press / tss, 3)

    best_ncomp <- model$bestTune$ncomp

    # 結果格納
    result_matrix[paste0("Best_ncomp_", target_var), file] <- best_ncomp
    result_matrix[paste0("R2_", target_var), file] <- R2
    result_matrix[paste0("RMSE_", target_var), file] <- RMSE_val
    result_matrix[paste0("MAE_", target_var), file] <- MAE_val
    result_matrix[paste0("RPD_", target_var), file] <- RPD_val
    result_matrix[paste0("Q2_", target_var), file] <- Q2_val
    result_matrix[paste0("n_samples_", target_var), file] <- nrow(df)
    result_matrix[paste0("n_features_", target_var), file] <- ncol(df) - 1

    # モデル＋データ保存
    model_data_bundle <- list(
      model = model,
      training_data = df
    )
    rds_file <- paste0("new20250624_model_data_", tools::file_path_sans_ext(file), "_", target_var, "_PLS_", today, ".rds")
    saveRDS(model_data_bundle, file = rds_file)

    # VIP保存
    vip <- calc_vip(pls_model, best_ncomp)
    vip_df <- data.frame(Variable = names(vip), VIP = vip)
    vip_file <- paste0("new20250625_vip_", tools::file_path_sans_ext(file), "_", target_var, "_PLS_", today, ".csv")
    write.csv(vip_df, vip_file, row.names = FALSE)

    # プロット
    range_max <- ceiling(max(c(obs, pred)) * 1.1)
    p <- ggplot(data.frame(Predicted = pred, Observed = obs), aes(x = Observed, y = Predicted)) +
      geom_point(color = "blue", alpha = 0.7) +
      geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "red") +
      scale_x_continuous(limits = c(0, range_max)) +
      scale_y_continuous(limits = c(0, range_max)) +
      coord_fixed(ratio = 1) +
      labs(
        title = paste0(target_var, " Prediction (", file, ")"),
        x = "Observed",
        y = "Predicted"
      ) +
      theme_minimal() +
      annotate("text", x = range_max * 0.05, y = range_max * 0.95, hjust = 0, vjust = 1, size = 4,
               label = paste0("RMSE = ", RMSE_val, "\nMAE = ", MAE_val, "\nRPD = ", RPD_val)) +
      annotate("text", x = range_max * 0.95, y = range_max * 0.05, hjust = 1, vjust = 0, size = 5,
               label = paste0("R² = ", R2, "\nQ² = ", Q2_val))

    plot_file <- paste0("new20250625_plot_", tools::file_path_sans_ext(file), "_", target_var, "_PLS_", today, ".pdf")
    pdf(plot_file, width = 6, height = 6)
    print(p)
    dev.off()
  }
}

# --- 結果出力 ---
output_file <- paste0("new20250625_pls_summary_", today, ".csv")
write.csv(result_matrix, output_file, row.names = TRUE)
cat("\nSummary saved as", output_file, "\n")



=== Processing: n_base.csv ===

---
Training model for: Jsc on n_base.csv 


Warning message:
"Removed 1 row containing missing values or values outside the scale range
(`geom_point()`)."



---
Training model for: Voc on n_base.csv 

---
Training model for: FF on n_base.csv 

---
Training model for: PCEmax on n_base.csv 


Warning message:
"Removed 1 row containing missing values or values outside the scale range
(`geom_point()`)."



=== Processing: n_base_OH.csv ===

---
Training model for: Jsc on n_base_OH.csv 


...


---
Training model for: FF on m_all_FP.csv 


Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: X17.1"
Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: X111.3"
Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: X46.4"



---
Training model for: PCEmax on m_all_FP.csv 


Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: X111.3"
Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: X46.4"
Warning message in preProcess.default(thresh = 0.95, k = 5, freqCut = 19, uniqueCut = 10, :
"These variables have zero variances: X17.1"
Warning message:
"Removed 16 rows containing missing values or values outside the scale range
(`geom_point()`)."



Summary saved as new20250625_pls_summary_20250626.csv 


In [5]:
print(vip)

          X    Egoptp1M     HOMOp1M     LUMOp1M Mmonomerp1M    MnkDap1M 
0.029289259 0.316828741 0.434289926 0.611324807 0.062820909 0.163768297 
   MwkDap1M      PDIp1M         X19         X22         X62         X66 
0.090724868 0.033949059 0.451605645 0.451605645 0.451605645 0.451605645 
        X93         X96        X105        X108        X112        X116 
0.451605645 0.451605645 0.451605645 0.451605645 0.451605645 0.451605645 
       X118        X123        X125        X126        X132        X144 
0.451605645 0.451605645 0.451605645 0.451605645 0.451605645 0.451605645 
       X145        X147        X154        X155        X157        X159 
0.451605645 0.451605645 0.451605645 0.451605645 0.451605645 0.451605645 
       X160        X162        X163        X164        X165       X17.1 
0.451605645 0.451605645 0.451605645 0.451605645 0.451605645 0.184288569 
      X19.1       X22.1       X26.1       X34.1       X36.1       X45.1 
0.323834335 0.323834335 0.236300250 0.116710064 0.1